# Download Whale NNUE models

Tải các file `.nnue` từ GitHub repo (raw file, không phải Release) về thư mục `models/`.
File đã tồn tại và đúng size thì bỏ qua.

In [ ]:
REPO = "niaowniaow/whale"
BRANCH = "main"
MODELS = ["whale_big.nnue"]
DEST_DIR = "models"

In [ ]:
import os
import urllib.request

os.makedirs(DEST_DIR, exist_ok=True)

def download(name):
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/models/{name}"
    dest = os.path.join(DEST_DIR, name)
    req = urllib.request.Request(url, headers={"User-Agent": "whale-dl"})
    with urllib.request.urlopen(req) as r:
        total = int(r.headers.get("Content-Length", 0))
        if os.path.exists(dest) and total and os.path.getsize(dest) == total:
            print(f"skip {name} (already {total} bytes)")
            return dest
        done = 0
        with open(dest, "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r{name}: {done}/{total} ({100*done/total:.1f}%)", end="")
        print(f"\ndone {name}: {os.path.getsize(dest)} bytes")
    return dest

for m in MODELS:
    download(m)

In [ ]:
import os
for m in MODELS:
    p = os.path.join(DEST_DIR, m)
    print(p, os.path.getsize(p) if os.path.exists(p) else "MISSING")

# Setup nnue-pytorch (chạy trên Kaggle)

Clone trainer + cài dependencies bằng shell.

In [ ]:
!test -d nnue-pytorch && git clone --depth 1 https://github.com/official-stockfish/nnue-pytorch.git || echo "nnue-pytorch already exists"
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r nnue-pytorch/requirements.txt

# Download training dataset (binpack)

Tải binpack từ HuggingFace `official-stockfish/master-binpacks`.
Đổi `DATA_URL`/`DATA_DEST` nếu muốn file khác.

In [ ]:
DATA_URL = "https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/test80-2022-08-aug-16tb7p.v6-dd.min.binpack"
DATA_DEST = "/tmp/data.binpack"

In [ ]:
import os
import urllib.request

parent = os.path.dirname(DATA_DEST)
if parent:
    os.makedirs(parent, exist_ok=True)

req = urllib.request.Request(DATA_URL, headers={"User-Agent": "whale-dl"})
with urllib.request.urlopen(req) as r:
    total = int(r.headers.get("Content-Length", 0))
    if os.path.exists(DATA_DEST) and total and os.path.getsize(DATA_DEST) == total:
        print(f"skip dataset (already {total} bytes)")
    else:
        done = 0
        with open(DATA_DEST, "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\rdataset: {done}/{total} ({100*done/total:.1f}%)", end="")
        print(f"\ndone dataset: {os.path.getsize(DATA_DEST)} bytes")